In [ ]:
!pip install yfinance scikit-learn seaborn tensorflow statsmodels

In [ ]:

import yfinance as yf
import pandas as pd 
import numpy as np 
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout
import matplotlib.pyplot as plt
import seaborn as sn
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_absolute_error
import joblib



In [ ]:
df = pd.read_csv(r'data/raw/data_2026_04_13.csv')

In [ ]:
df.columns = df.columns.get_level_values(0)
df = df.iloc[2:]

In [ ]:
df = df.rename(columns={'Price': 'data', 'Close': 'fechamento', 'High':'maxima', 'Low': 'minima', 'Open': 'abertura', 'Volume': 'volume'})


In [ ]:
df = df[df['data'] > '2021-01-01']

In [ ]:
df.head()

In [ ]:
df["data"] = pd.to_datetime(df["data"])

numeric_cols = ["fechamento", "maxima", "minima", "abertura", "volume"]
df[numeric_cols] = df[numeric_cols].astype(float)

In [ ]:
df.dtypes

In [ ]:
df['return'] = df['fechamento'].pct_change()
df['range'] = df['maxima'] - df['minima']
df['variacao'] = df['fechamento'] - df['abertura']
df['vol_10'] = df['return'].rolling(10).std()
df['vol_20'] = df['return'].rolling(20).std()
df['fechamento_1d'] = df['fechamento'].shift(1)
df['fechamento_2d'] = df['fechamento'].shift(2)
df['fechamento_3d'] = df['fechamento'].shift(3)
df['sma_5'] = df['fechamento'].rolling(5).mean()
df['sma_10'] = df['fechamento'].rolling(10).mean()
df['sma_20'] = df['fechamento'].rolling(20).mean()
df['ema_5'] = df['fechamento'].ewm(span=5).mean()
df['ema_10'] = df['fechamento'].ewm(span=10).mean()
df['mom_5'] = df['fechamento'] - df['fechamento'].shift(5)
df['mom_10'] = df['fechamento'] - df['fechamento'].shift(10)
df['target'] = df['fechamento'].shift(-1)

delta = df['fechamento'].diff()

gain = (delta.where(delta > 0, 0)).rolling(14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(14).mean()

rs = gain / loss
df['rsi'] = 100 - (100 / (1 + rs))

df = df.dropna()

In [ ]:
(df.isna().sum() / df.shape[0] * 100).sort_values(ascending = False)

In [ ]:
df.describe().transpose()

In [ ]:
cols = df.select_dtypes(include='number').columns

ncols = 4
nrows = (len(cols) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4*nrows))
axes = axes.flatten()

for ax, col in zip(axes, cols):
    df[col].plot(kind='box', ax=ax)
    ax.set_title(col)

for ax in axes[len(cols):]
    ax.remove()

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(16,8))
plt.title('Variação Fechamento')

plt.plot(df['data'], df['fechamento'])
plt.xticks(rotation=45) 
plt.show()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df[['fechamento', 'volume', 'sma_5', 'sma_10', 'vol_10', 'mom_5', 'target', 'maxima', 'minima', 'variacao', 'abertura']].head()

In [ ]:
# Preparar dados para o modelo LSTM
features = df.drop(['data', 'target'], axis=1).values
target = df['target'].values

# Normalizar os dados
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# Dividir em treino e teste
X_train, X_test, y_train, y_test = train_test_split(features_scaled, target, test_size=0.2, random_state=42)

# Reshape para LSTM
X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

In [ ]:
# Construir modelo LSTM
model = Sequential([
    LSTM(64, activation='relu', input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

In [ ]:
# Treinar o modelo
early_stop = EarlyStopping(monitor='val_loss', patience=10)
history = model.fit(X_train, y_train, epochs=100, batch_size=32, validation_split=0.2, callbacks=[early_stop], verbose=1)

In [ ]:
# Avaliar o modelo
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f"MAE do modelo: {mae}")

In [ ]:
# Visualizar histórico de treino
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['train', 'val'])

plt.subplot(1, 2, 2)
plt.plot(history.history['mae'])
plt.plot(history.history['val_mae'])
plt.title('MAE')
plt.ylabel('MAE')
plt.xlabel('Epoch')
plt.legend(['train', 'val'])

plt.tight_layout()
plt.show()

In [ ]:
# Salvar o modelo
model.save('models/lstm_model.h5')
joblib.dump(scaler, 'models/scaler.pkl')
print("Modelo e scaler salvos com sucesso!")

In [ ]:
# Gráfico de comparação: real vs predito
plt.figure(figsize=(12, 6))
plt.plot(y_test, label='Real', alpha=0.7)
plt.plot(y_pred, label='Predito', alpha=0.7)
plt.title('Comparação: Valores Reais vs Preditos')
plt.xlabel('Amostra')
plt.ylabel('Preço')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("std real:", np.std(y_test))
print("std pred:", np.std(y_pred))